In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter

# Sample text
text = """
I love deep learning.
I am learning simple rnn.
Deep learning is interesting.
Simple rnn is easy to understand.
"""

In [ ]:
# Preprocess
sentences = [line.strip().lower() for line in text.split('.') if line.strip()]
print("sentences:",sentences)
words = [w for line in sentences for w in line.split()]
print("words:",words)
# Build vocabulary
vocab = ['<pad>'] + sorted(set(words))
print("vocab:",vocab)
word2idx = {w: i for i, w in enumerate(vocab)}
print("word2idx:",word2idx)
idx2word = {i: w for w, i in word2idx.items()}
print("idx2word:",idx2word)


sentences: ['i love deep learning', 'i am learning simple rnn', 'deep learning is interesting', 'simple rnn is easy to understand']
words: ['i', 'love', 'deep', 'learning', 'i', 'am', 'learning', 'simple', 'rnn', 'deep', 'learning', 'is', 'interesting', 'simple', 'rnn', 'is', 'easy', 'to', 'understand']
vocab: ['<pad>', 'am', 'deep', 'easy', 'i', 'interesting', 'is', 'learning', 'love', 'rnn', 'simple', 'to', 'understand']
word2idx: {'<pad>': 0, 'am': 1, 'deep': 2, 'easy': 3, 'i': 4, 'interesting': 5, 'is': 6, 'learning': 7, 'love': 8, 'rnn': 9, 'simple': 10, 'to': 11, 'understand': 12}
idx2word: {0: '<pad>', 1: 'am', 2: 'deep', 3: 'easy', 4: 'i', 5: 'interesting', 6: 'is', 7: 'learning', 8: 'love', 9: 'rnn', 10: 'simple', 11: 'to', 12: 'understand'}


In [ ]:
# Build sequences
sequences = []
for line in sentences:
    token_ids = [word2idx[w] for w in line.split()]
    for i in range(1, len(token_ids)):
        sequences.append(token_ids[: i + 1])
print("sequences:",sequences)

# Pad sequences
max_len = max(len(seq) for seq in sequences)
padded = [
    [0] * (max_len - len(seq)) + seq
    for seq in sequences
]
print("padded:",padded)

X = [seq[:-1] for seq in padded]
y = [seq[-1] for seq in padded]
print("X:",X)
print("y:",y)

# Defines a custom dataset class for PyTorch and prepares the data for training
class NextWordDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

dataset = NextWordDataset(X, y)

loader = DataLoader(dataset, batch_size=4, shuffle=True)


sequences: [[4, 8], [4, 8, 2], [4, 8, 2, 7], [4, 1], [4, 1, 7], [4, 1, 7, 10], [4, 1, 7, 10, 9], [2, 7], [2, 7, 6], [2, 7, 6, 5], [10, 9], [10, 9, 6], [10, 9, 6, 3], [10, 9, 6, 3, 11], [10, 9, 6, 3, 11, 12]]
padded: [[0, 0, 0, 0, 4, 8], [0, 0, 0, 4, 8, 2], [0, 0, 4, 8, 2, 7], [0, 0, 0, 0, 4, 1], [0, 0, 0, 4, 1, 7], [0, 0, 4, 1, 7, 10], [0, 4, 1, 7, 10, 9], [0, 0, 0, 0, 2, 7], [0, 0, 0, 2, 7, 6], [0, 0, 2, 7, 6, 5], [0, 0, 0, 0, 10, 9], [0, 0, 0, 10, 9, 6], [0, 0, 10, 9, 6, 3], [0, 10, 9, 6, 3, 11], [10, 9, 6, 3, 11, 12]]
X: [[0, 0, 0, 0, 4], [0, 0, 0, 4, 8], [0, 0, 4, 8, 2], [0, 0, 0, 0, 4], [0, 0, 0, 4, 1], [0, 0, 4, 1, 7], [0, 4, 1, 7, 10], [0, 0, 0, 0, 2], [0, 0, 0, 2, 7], [0, 0, 2, 7, 6], [0, 0, 0, 0, 10], [0, 0, 0, 10, 9], [0, 0, 10, 9, 6], [0, 10, 9, 6, 3], [10, 9, 6, 3, 11]]
y: [8, 2, 7, 1, 7, 10, 9, 7, 6, 5, 9, 6, 3, 11, 12]


In [ ]:
# Model
class SimpleRNNModel(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size, padding_idx=0)
        self.rnn = nn.RNN(embed_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.rnn(emb)
        out = out[:, -1, :]
        return self.fc(out)

model = SimpleRNNModel(vocab_size=len(vocab), embed_size=50, hidden_size=100)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

In [ ]:
# Train
for epoch in range(100):
    total_loss = 0.0
    for xb, yb in loader:
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, loss = {total_loss / len(loader):.4f}")

# Predict
def predict_next(seed_text, next_words=1):
    model.eval()
    words = seed_text.lower().split()
    for _ in range(next_words):
        token_ids = [word2idx.get(word, 0) for word in words]
        token_ids = [0] * (max_len - 1 - len(token_ids)) + token_ids
        inp = torch.tensor([token_ids], dtype=torch.long)
        with torch.no_grad():
            logits = model(inp)
            pred = torch.argmax(F.softmax(logits, dim=-1), dim=-1).item()
        words.append(idx2word[pred])
    return ' '.join(words)

print(predict_next("learning ", next_words=2))
print(predict_next("i love", next_words=2))
print(predict_next("deep learning", next_words=2))

Epoch 20, loss = 0.1227
Epoch 40, loss = 0.0905
Epoch 60, loss = 0.0923
Epoch 80, loss = 0.1045
Epoch 100, loss = 0.0972
learning is interesting
i love deep learning
deep learning is interesting
